In [63]:
# ============================================================
# CELL 1: imports + project paths
# ============================================================

import os
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from pybaseball import playerid_lookup

warnings.filterwarnings("ignore")

# ----------------------------
# Base paths
# ----------------------------
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

STATCAST_MAIN_PATH = DATA_DIR / "statcast_2021_2025.parquet"
STATCAST_YTD_PATH = DATA_DIR / "statcast_2026_ytd.parquet"

PA_MODEL_PATH = MODELS_DIR / "pregame_pa_model.cbm"

# try a few possible meta file names
POSSIBLE_META_PATHS = [
    MODELS_DIR / "pregame_pa_model_meta.json",
    MODELS_DIR / "model_meta.json",
    MODELS_DIR / "winloss_pa_model_meta.json"
]

PA_META_PATH = None
for p in POSSIBLE_META_PATHS:
    if p.exists():
        PA_META_PATH = p
        break

print("BASE_DIR:", BASE_DIR.resolve())
print("Main statcast file exists:", STATCAST_MAIN_PATH.exists())
print("YTD statcast file exists:", STATCAST_YTD_PATH.exists())
print("Pregame PA model exists:", PA_MODEL_PATH.exists())
print("Resolved meta path:", PA_META_PATH if PA_META_PATH else "None found")

BASE_DIR: C:\Users\Matthew\OneDrive\Desktop\machine learning- baseball
Main statcast file exists: True
YTD statcast file exists: True
Pregame PA model exists: True
Resolved meta path: models\model_meta.json


In [64]:
# ============================================================
# CELL 2: load model, metadata, and statcast data
# ============================================================

# ----------------------------
# Load pregame PA model
# ----------------------------
pa_model = CatBoostClassifier()
pa_model.load_model(str(PA_MODEL_PATH))
print("Loaded PA model.")

# ----------------------------
# Load metadata if available
# ----------------------------
if PA_META_PATH is not None:
    with open(PA_META_PATH, "r") as f:
        pa_meta = json.load(f)
    print("Loaded PA meta from:", PA_META_PATH)
    print("Meta keys:", list(pa_meta.keys())[:15])
else:
    pa_meta = {}
    print("No meta json found. Continuing with empty pa_meta.")

# ----------------------------
# Load statcast data
# ----------------------------
df_main = pd.read_parquet(STATCAST_MAIN_PATH)
print("df_main shape:", df_main.shape)

if STATCAST_YTD_PATH.exists():
    df_ytd = pd.read_parquet(STATCAST_YTD_PATH)
    print("df_ytd shape:", df_ytd.shape)

    df = pd.concat([df_main, df_ytd], ignore_index=True)
    print("Combined df shape:", df.shape)
else:
    df_ytd = pd.DataFrame()
    df = df_main.copy()
    print("No YTD file found. Using df_main only.")

print("\nSample columns:")
print(df.columns.tolist()[:40])

Loaded PA model.
Loaded PA meta from: models\model_meta.json
Meta keys: ['features', 'cat_cols']
df_main shape: (3846144, 119)
df_ytd shape: (0, 0)
Combined df shape: (3846144, 119)

Sample columns:
['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated']


In [65]:
# ============================================================
# CELL 3: basic cleaning + required column check
# ============================================================

df = df.copy()

# Convert game date
if "game_date" in df.columns:
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

# Drop fully empty rows
df = df.dropna(how="all").reset_index(drop=True)

required_cols = [
    "game_date",
    "game_pk",
    "inning",
    "inning_topbot",
    "batter",
    "pitcher",
    "events",
    "stand",
    "p_throws",
    "home_team",
    "away_team"
]

missing_cols = [c for c in required_cols if c not in df.columns]

print("Cleaned df shape:", df.shape)
print("Missing required columns:", missing_cols if missing_cols else "None")

print("\nA few useful columns that DO exist:")
existing_preview = [c for c in [
    "game_date", "game_pk", "inning", "inning_topbot", "at_bat_number",
    "pitch_number", "batter", "pitcher", "events", "stand", "p_throws",
    "home_team", "away_team", "post_home_score", "post_away_score"
] if c in df.columns]
print(existing_preview)

Cleaned df shape: (3846144, 119)
Missing required columns: None

A few useful columns that DO exist:
['game_date', 'game_pk', 'inning', 'inning_topbot', 'at_bat_number', 'pitch_number', 'batter', 'pitcher', 'events', 'stand', 'p_throws', 'home_team', 'away_team', 'post_home_score', 'post_away_score']


In [66]:
# ============================================================
# CELL 4: PA outcome helper definitions
# ============================================================

PA_CLASSES = ["out", "strikeout", "walk", "single", "double", "triple", "home_run"]

BB_EVENTS = {"walk", "intent_walk"}
K_EVENTS = {"strikeout", "strikeout_double_play"}

def map_event_to_pa_class(event):
    if pd.isna(event):
        return "out"

    event = str(event).lower().strip()

    if event in BB_EVENTS:
        return "walk"
    if event in K_EVENTS:
        return "strikeout"
    if event == "single":
        return "single"
    if event == "double":
        return "double"
    if event == "triple":
        return "triple"
    if event == "home_run":
        return "home_run"

    return "out"

df["pa_class_simple"] = df["events"].apply(map_event_to_pa_class)

print(df["pa_class_simple"].value_counts(dropna=False))

pa_class_simple
out          3312910
strikeout     228458
single        141595
walk           84804
double         43720
home_run       30918
triple          3739
Name: count, dtype: int64


In [67]:
# ============================================================
# NEW PA DATASET: one row per true plate appearance
# ============================================================

print("Building true PA dataset...")

pa_sort_cols = [c for c in ["game_pk", "at_bat_number", "pitch_number"] if c in df.columns]
pa_work = df.copy()

if pa_sort_cols:
    pa_work = pa_work.sort_values(pa_sort_cols)

# Keep last pitch of each at-bat
if "game_pk" in pa_work.columns and "at_bat_number" in pa_work.columns:
    pa_df = (
        pa_work.groupby(["game_pk", "at_bat_number"], as_index=False)
        .tail(1)
        .copy()
    )
else:
    # fallback if at_bat_number missing
    pa_df = pa_work[pa_work["events"].notna()].copy()

# Keep only rows that have batter, pitcher, and event context
pa_df = pa_df[
    pa_df["batter"].notna() &
    pa_df["pitcher"].notna()
].copy()

# Map PA outcome
pa_df["pa_class_simple"] = pa_df["events"].apply(map_event_to_pa_class)

print("PA df shape:", pa_df.shape)
print("\nLeague PA distribution:")
print(pa_df["pa_class_simple"].value_counts(normalize=True).sort_index())

Building true PA dataset...
PA df shape: (1005300, 120)

League PA distribution:
pa_class_simple
double       0.043490
home_run     0.030755
out          0.469577
single       0.140849
strikeout    0.227254
triple       0.003719
walk         0.084357
Name: proportion, dtype: float64


In [68]:
# ============================================================
# CELL 5 (FIXED): build starter usage table
# ============================================================

work = df.copy()

# Identify which team is pitching
work["pitching_team"] = np.where(
    work["inning_topbot"].astype(str).str.lower().eq("top"),
    work["home_team"],
    work["away_team"]
)

# Sort safely
sort_cols = [c for c in ["game_date","game_pk","inning","at_bat_number","pitch_number"] if c in work.columns]
if sort_cols:
    work = work.sort_values(sort_cols)

# Identify starters (first pitcher per team/game)
starter_rows = (
    work.groupby(["game_pk","pitching_team"], as_index=False)
    .first()[["game_pk","pitching_team","pitcher"]]
    .rename(columns={"pitcher":"starter_pitcher_id"})
)

work = work.merge(
    starter_rows,
    on=["game_pk","pitching_team"],
    how="left"
)

work["is_starter_pitcher"] = work["pitcher"] == work["starter_pitcher_id"]

starter_df = work[work["is_starter_pitcher"]].copy()

# ----------------------------
# Estimate pitches thrown
# ----------------------------
# Count rows as pitch events (Statcast rows are pitches)

starter_pitch_counts = (
    starter_df.groupby(["game_pk","pitcher"])
    .size()
    .reset_index(name="starter_pitches")
)

# Batters faced
starter_bf = (
    starter_df.groupby(["game_pk","pitcher"])["batter"]
    .nunique()
    .reset_index(name="batters_faced")
)

# Innings seen
starter_innings = (
    starter_df.groupby(["game_pk","pitcher"])["inning"]
    .nunique()
    .reset_index(name="innings_seen")
)

starter_game_summary = starter_pitch_counts.merge(starter_bf,on=["game_pk","pitcher"])
starter_game_summary = starter_game_summary.merge(starter_innings,on=["game_pk","pitcher"])

starter_usage = (
    starter_game_summary.groupby("pitcher",as_index=False)
    .agg(
        starts=("game_pk","nunique"),
        avg_batters_faced=("batters_faced","mean"),
        avg_innings_seen=("innings_seen","mean"),
        avg_pitches=("starter_pitches","mean"),
        med_pitches=("starter_pitches","median"),
        max_pitches=("starter_pitches","max")
    )
)

starter_usage = starter_usage.sort_values(
    ["starts","avg_pitches"],
    ascending=[False,False]
).reset_index(drop=True)

print("starter_usage shape:",starter_usage.shape)
display(starter_usage.head(15))

starter_usage shape: (946, 7)


,pitcher,starts,avg_batters_faced,avg_innings_seen,avg_pitches,med_pitches,max_pitches
0,656302,177,9.090395,5.564972,92.412429,96.0,115
1,592332,173,9.167630,5.959538,92.150289,96.0,118
2,657277,171,9.198830,6.222222,91.192982,95.0,112
3,621244,170,9.047059,5.905882,87.723529,90.0,109
4,605400,169,9.195266,6.029586,91.739645,95.0,117
5,554430,165,9.242424,6.321212,95.484848,98.0,118
6,622491,165,9.121212,5.909091,93.836364,97.0,123
7,450203,164,9.103659,5.518293,88.195122,94.0,112
8,605135,163,9.067485,5.932515,92.134969,95.0,115
9,668678,163,9.098160,5.858896,91.822086,96.0,115


In [69]:
# ============================================================
# CELL 6 (FIXED): build reliever usage table
# ============================================================

relief_df = work[~work["is_starter_pitcher"]].copy()

# Count pitches
reliever_pitch_counts = (
    relief_df.groupby(["game_pk","pitcher"])
    .size()
    .reset_index(name="relief_pitches")
)

# Batters faced
reliever_bf = (
    relief_df.groupby(["game_pk","pitcher"])["batter"]
    .nunique()
    .reset_index(name="batters_faced")
)

# Entry inning
reliever_innings = (
    relief_df.groupby(["game_pk","pitcher"])
    .agg(
        innings_seen=("inning","nunique"),
        first_inning_entered=("inning","min")
    )
    .reset_index()
)

reliever_game_summary = reliever_pitch_counts.merge(reliever_bf,on=["game_pk","pitcher"])
reliever_game_summary = reliever_game_summary.merge(reliever_innings,on=["game_pk","pitcher"])

reliever_usage = (
    reliever_game_summary.groupby("pitcher",as_index=False)
    .agg(
        relief_apps=("game_pk","nunique"),
        avg_relief_innings=("innings_seen","mean"),
        avg_relief_bf=("batters_faced","mean"),
        avg_relief_pitches=("relief_pitches","mean"),
        avg_entry_inning=("first_inning_entered","mean"),
        median_entry_inning=("first_inning_entered","median")
    )
)

def assign_reliever_role(avg_entry_inning):
    if pd.isna(avg_entry_inning):
        return "middle"
    if avg_entry_inning >= 8.5:
        return "closer"
    if avg_entry_inning >= 7.5:
        return "setup"
    if avg_entry_inning <= 5.0:
        return "long"
    return "middle"

reliever_usage["role_guess"] = reliever_usage["avg_entry_inning"].apply(assign_reliever_role)

print("reliever_usage shape:",reliever_usage.shape)
print(reliever_usage["role_guess"].value_counts())
display(reliever_usage.head(15))

reliever_usage shape: (2567, 8)
role_guess
middle    1420
setup      503
closer     324
long       320
Name: count, dtype: int64


,pitcher,relief_apps,avg_relief_innings,avg_relief_bf,avg_relief_pitches,avg_entry_inning,median_entry_inning,role_guess
0,405395,1,1.000000,7.000000,27.000000,9.0,9.0,closer
1,424144,22,1.090909,3.727273,8.772727,6.772727,6.0,middle
2,425844,5,3.200000,8.600000,48.200000,4.0,4.0,long
3,425877,2,1.000000,6.000000,20.000000,8.5,8.5,closer
4,429722,39,1.923077,6.000000,26.307692,6.794872,8.0,middle
5,431148,2,1.500000,8.000000,23.500000,5.5,5.5,middle
6,433589,82,1.207317,4.048780,13.621951,7.134146,7.0,middle
7,434538,3,1.333333,4.666667,15.666667,6.666667,7.0,middle
8,444468,4,1.000000,3.500000,14.250000,7.5,7.0,setup
9,444482,1,1.000000,9.000000,28.000000,9.0,9.0,closer


In [70]:
# ============================================================
# CELL 7: sanity checks
# ============================================================

print("Starter avg pitches summary:")
print(starter_usage["avg_pitches"].describe())

print("\nStarter avg innings summary:")
print(starter_usage["avg_innings_seen"].describe())

print("\nReliever avg pitches summary:")
print(reliever_usage["avg_relief_pitches"].describe())

print("\nReliever avg entry inning summary:")
print(reliever_usage["avg_entry_inning"].describe())

Starter avg pitches summary:
count    946.000000
mean      60.331722
std       25.935611
min        4.000000
25%       34.250000
50%       70.600000
75%       82.136340
max       98.526316
Name: avg_pitches, dtype: float64

Starter avg innings summary:
count    946.000000
mean       3.887408
std        1.595389
min        1.000000
25%        2.500000
50%        4.408333
75%        5.250000
max        7.000000
Name: avg_innings_seen, dtype: float64

Reliever avg pitches summary:
count    2567.000000
mean       19.648172
std        12.191787
min         1.000000
25%        12.557919
50%        17.000000
75%        24.000000
max        96.000000
Name: avg_relief_pitches, dtype: float64

Reliever avg entry inning summary:
count      2567.0
mean     6.805689
std      1.539085
min           1.0
25%      6.134848
50%           7.0
75%           7.8
max          14.0
Name: avg_entry_inning, dtype: Float64


In [71]:
# ============================================================
# CELL 8: build player lookup tables
# ============================================================

print("Building player lookup tables...")

# Get unique batters and pitchers from statcast
unique_batters = df["batter"].dropna().unique()
unique_pitchers = df["pitcher"].dropna().unique()

player_ids = np.unique(np.concatenate([unique_batters, unique_pitchers]))

player_lookup = pd.DataFrame({"mlbam_id": player_ids})

print("Unique players in dataset:", len(player_lookup))

player_lookup.head()

Building player lookup tables...
Unique players in dataset: 5285


,mlbam_id
0,405395
1,408234
2,424144
3,425772
4,425784


In [85]:
# ============================================================
# ROBUST PLAYER LOOKUP WITH MANUAL OVERRIDES
# ============================================================

import unicodedata
from pybaseball import playerid_lookup

# Manual overrides for names that pybaseball misses
# You can add more here anytime
MANUAL_ID_MAP = {
    "teoscar hernandez": 606192,
    "kike hernandez": 571771,
    "enrique hernandez": 571771,
}

def normalize_name(name):
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("utf-8")
    name = name.lower().strip()
    return " ".join(name.split())

def get_player_id_from_full_name(full_name):
    clean = normalize_name(full_name)

    # 1) manual override first
    if clean in MANUAL_ID_MAP:
        return MANUAL_ID_MAP[clean]

    # 2) fallback to pybaseball lookup
    parts = full_name.strip().split()
    if len(parts) < 2:
        print("Bad name format:", full_name)
        return None

    first = parts[0]
    last = parts[-1]

    try:
        res = playerid_lookup(last, first)
    except Exception as e:
        print("Lookup failed:", full_name, "|", e)
        return None

    if len(res) == 0:
        print("Player not found:", full_name)
        return None

    if "mlb_played_last" in res.columns:
        res = res.sort_values("mlb_played_last", ascending=False)

    return int(res.iloc[0]["key_mlbam"])

In [86]:
# ============================================================
# LINEUP -> IDS
# ============================================================

def lineup_to_ids(lineup):
    ids = []

    for player in lineup:
        pid = get_player_id_from_full_name(player)

        if pid is None:
            print("FAILED:", player)

        ids.append(pid)

    return ids

In [74]:
# ============================================================
# CELL 11 (FIXED): build stabilized batter/pitcher outcome tables
# ============================================================

# League average rates
league_rates = pa_df["pa_class_simple"].value_counts(normalize=True).to_dict()
for c in PA_CLASSES:
    league_rates[c] = league_rates.get(c, 0.0)

league_row = pd.Series(league_rates)

# Batter outcome counts
batter_counts = (
    pa_df.groupby("batter")["pa_class_simple"]
    .value_counts()
    .unstack(fill_value=0)
)

for c in PA_CLASSES:
    if c not in batter_counts.columns:
        batter_counts[c] = 0

batter_pa = batter_counts.sum(axis=1)

# Shrink toward league average
BATTER_PRIOR = 200

batter_stats = batter_counts.copy()
for c in PA_CLASSES:
    batter_stats[c] = (
        batter_counts[c] + BATTER_PRIOR * league_rates[c]
    ) / (batter_pa + BATTER_PRIOR)

batter_stats = batter_stats.reset_index()

# Pitcher outcome counts allowed
pitcher_counts = (
    pa_df.groupby("pitcher")["pa_class_simple"]
    .value_counts()
    .unstack(fill_value=0)
)

for c in PA_CLASSES:
    if c not in pitcher_counts.columns:
        pitcher_counts[c] = 0

pitcher_pa = pitcher_counts.sum(axis=1)

PITCHER_PRIOR = 250

pitcher_stats = pitcher_counts.copy()
for c in PA_CLASSES:
    pitcher_stats[c] = (
        pitcher_counts[c] + PITCHER_PRIOR * league_rates[c]
    ) / (pitcher_pa + PITCHER_PRIOR)

pitcher_stats = pitcher_stats.reset_index()

print("Built batter_stats:", batter_stats.shape)
print("Built pitcher_stats:", pitcher_stats.shape)
print("League rates:", league_rates)

Built batter_stats: (3243, 8)
Built pitcher_stats: (2663, 8)
League rates: {'out': 0.46957724062468914, 'strikeout': 0.22725355615239232, 'single': 0.14084850293444742, 'walk': 0.08435690838555655, 'double': 0.04348950562021287, 'home_run': 0.030754998507908086, 'triple': 0.003719287774793594}


In [75]:
# ============================================================
# CELL 12 (FIXED): helper functions for matchup probabilities
# ============================================================

batter_stats_indexed = batter_stats.set_index("batter")
pitcher_stats_indexed = pitcher_stats.set_index("pitcher")

def safe_prob_vector(prob_dict):
    arr = np.array([prob_dict.get(c, 0.0) for c in PA_CLASSES], dtype=float)

    if arr.sum() <= 0:
        arr = np.array([league_rates[c] for c in PA_CLASSES], dtype=float)

    arr = np.clip(arr, 1e-9, None)
    arr = arr / arr.sum()
    return arr

def get_batter_probs(batter_id):
    if batter_id in batter_stats_indexed.index:
        row = batter_stats_indexed.loc[batter_id, PA_CLASSES]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return safe_prob_vector(row.to_dict())
    return safe_prob_vector(league_rates)

def get_pitcher_probs(pitcher_id):
    if pitcher_id in pitcher_stats_indexed.index:
        row = pitcher_stats_indexed.loc[pitcher_id, PA_CLASSES]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return safe_prob_vector(row.to_dict())
    return safe_prob_vector(league_rates)

def get_matchup_probs(batter_id, pitcher_id):
    b_probs = get_batter_probs(batter_id)
    p_probs = get_pitcher_probs(pitcher_id)
    l_probs = np.array([league_rates[c] for c in PA_CLASSES], dtype=float)

    # multiplicative adjustment around league mean
    probs = l_probs * (b_probs / l_probs) * (p_probs / l_probs)

    probs = np.clip(probs, 1e-9, None)
    probs = probs / probs.sum()
    return probs

In [76]:
# ============================================================
# SANITY CHECK: baseline matchup rates
# ============================================================

print("League rates:")
for c in PA_CLASSES:
    print(f"{c}: {league_rates[c]:.4f}")

test_probs = get_matchup_probs(592450, 543037)  # Aaron Judge vs Gerrit Cole example format
print("\nExample matchup probs:")
for c, p in zip(PA_CLASSES, test_probs):
    print(f"{c}: {p:.4f}")

League rates:
out: 0.4696
strikeout: 0.2273
walk: 0.0844
single: 0.1408
double: 0.0435
triple: 0.0037
home_run: 0.0308

Example matchup probs:
out: 0.3244
strikeout: 0.3227
walk: 0.1201
single: 0.1175
double: 0.0366
triple: 0.0008
home_run: 0.0779


In [77]:
# ============================================================
# CELL 13: simulate one plate appearance
# ============================================================

def simulate_pa(batter_id, pitcher_id):
    probs = get_matchup_probs(batter_id, pitcher_id)
    outcome = np.random.choice(PA_CLASSES, p=probs)
    return outcome

In [78]:
# ============================================================
# CELL 14: base advancement logic
# ============================================================

def advance_runners(bases, outcome):
    """
    bases = [on_1b, on_2b, on_3b] where values are 0 or 1
    returns updated_bases, runs_scored, outs_added
    """
    on_1b, on_2b, on_3b = bases
    runs = 0
    outs = 0

    if outcome in ["out", "strikeout"]:
        outs = 1
        return [on_1b, on_2b, on_3b], runs, outs

    if outcome == "walk":
        if on_1b and on_2b and on_3b:
            runs += 1
            return [1, 1, 1], runs, outs
        elif on_1b and on_2b:
            return [1, 1, 1], runs, outs
        elif on_1b:
            return [1, 1, on_3b], runs, outs
        else:
            return [1, on_2b, on_3b], runs, outs

    if outcome == "single":
        runs += on_3b
        new_on_3b = on_2b
        new_on_2b = on_1b
        new_on_1b = 1
        return [new_on_1b, new_on_2b, new_on_3b], runs, outs

    if outcome == "double":
        runs += on_3b + on_2b
        new_on_3b = on_1b
        new_on_2b = 1
        new_on_1b = 0
        return [new_on_1b, new_on_2b, new_on_3b], runs, outs

    if outcome == "triple":
        runs += on_1b + on_2b + on_3b
        return [0, 0, 1], runs, outs

    if outcome == "home_run":
        runs += on_1b + on_2b + on_3b + 1
        return [0, 0, 0], runs, outs

    outs = 1
    return [on_1b, on_2b, on_3b], runs, outs

In [79]:
# ============================================================
# CELL 15: simulate one half inning
# ============================================================

def simulate_half_inning(lineup_ids, pitcher_id, lineup_pos=0):
    outs = 0
    runs = 0
    bases = [0, 0, 0]

    while outs < 3:
        batter_id = lineup_ids[lineup_pos % 9]
        outcome = simulate_pa(batter_id, pitcher_id)

        bases, new_runs, new_outs = advance_runners(bases, outcome)

        runs += new_runs
        outs += new_outs
        lineup_pos += 1

    return runs, lineup_pos

In [80]:
# ============================================================
# CELL 16 (FIXED): simulate one full game with extras
# ============================================================

def simulate_game(away_lineup_ids, home_lineup_ids, away_pitcher_id, home_pitcher_id, max_extra_innings=6):
    away_runs = 0
    home_runs = 0

    away_lineup_pos = 0
    home_lineup_pos = 0

    # first 9 innings
    for inning in range(1, 10):
        r, away_lineup_pos = simulate_half_inning(
            away_lineup_ids,
            home_pitcher_id,
            away_lineup_pos
        )
        away_runs += r

        r, home_lineup_pos = simulate_half_inning(
            home_lineup_ids,
            away_pitcher_id,
            home_lineup_pos
        )
        home_runs += r

    # extras until winner
    extra_inning = 10
    extras_used = 0

    while away_runs == home_runs and extras_used < max_extra_innings:
        r, away_lineup_pos = simulate_half_inning(
            away_lineup_ids,
            home_pitcher_id,
            away_lineup_pos
        )
        away_runs += r

        r, home_lineup_pos = simulate_half_inning(
            home_lineup_ids,
            away_pitcher_id,
            home_lineup_pos
        )
        home_runs += r

        extra_inning += 1
        extras_used += 1

    return {
        "away_runs": away_runs,
        "home_runs": home_runs
    }

In [81]:
# ============================================================
# CELL 17 (FIXED): simulate many games
# ============================================================

def simulate_matchup(away_lineup_ids, home_lineup_ids, away_pitcher_id, home_pitcher_id, n_sims=1000):
    results = []

    for _ in range(n_sims):
        g = simulate_game(
            away_lineup_ids=away_lineup_ids,
            home_lineup_ids=home_lineup_ids,
            away_pitcher_id=away_pitcher_id,
            home_pitcher_id=home_pitcher_id
        )
        results.append(g)

    results_df = pd.DataFrame(results)

    away_avg = results_df["away_runs"].mean()
    home_avg = results_df["home_runs"].mean()

    away_wins = (results_df["away_runs"] > results_df["home_runs"]).mean()
    home_wins = (results_df["home_runs"] > results_df["away_runs"]).mean()
    unresolved_ties = (results_df["home_runs"] == results_df["away_runs"]).mean()

    return {
        "results_df": results_df,
        "away_avg_runs": away_avg,
        "home_avg_runs": home_avg,
        "away_win_pct": away_wins,
        "home_win_pct": home_wins,
        "unresolved_tie_pct": unresolved_ties
    }

In [82]:
# ============================================================
# CELL 18 (FIXED): display matchup summary
# ============================================================

def print_matchup_summary(sim_result, away_team="Away", home_team="Home"):
    print(f"{away_team} avg runs: {sim_result['away_avg_runs']:.2f}")
    print(f"{home_team} avg runs: {sim_result['home_avg_runs']:.2f}")
    print()
    print(f"{away_team} win %: {sim_result['away_win_pct']*100:.1f}%")
    print(f"{home_team} win %: {sim_result['home_win_pct']*100:.1f}%")
    print(f"Unresolved tie %: {sim_result['unresolved_tie_pct']*100:.1f}%")

In [96]:
# ============================================================
# CELL 19: test matchup
# ============================================================

away_lineup = [
"Aaron Judge",
"Juan Soto",
"Giancarlo Stanton",
"Gleyber Torres",
"Anthony Rizzo",
"Alex Verdugo",
"Austin Wells",
"Anthony Volpe",
"Trent Grisham"
]

home_lineup = [
    "Mookie Betts",
    "Freddie Freeman",
    "Will Smith",
    "Max Muncy",
    "Teoscar Hernandez",
    "Gavin Lux",
    "James Outman",
    "Chris Taylor",
    "Enrique Hernandez"
]

home_lineup_ids = lineup_to_ids(home_lineup)
print(home_lineup_ids)
validate_lineup_ids(home_lineup_ids, "Home")

away_pitcher_id = get_player_id("Gerrit", "Cole")
home_pitcher_id = get_player_id("Yoshinobu", "Yamamoto")

print("Away lineup IDs:", away_lineup_ids)
print("Home lineup IDs:", home_lineup_ids)
print("Away pitcher ID:", away_pitcher_id)
print("Home pitcher ID:", home_pitcher_id)

sim_result = simulate_matchup(
    away_lineup_ids=away_lineup_ids,
    home_lineup_ids=home_lineup_ids,
    away_pitcher_id=away_pitcher_id,
    home_pitcher_id=home_pitcher_id,
    n_sims=1000
)

print_matchup_summary(sim_result, away_team="Yankees", home_team="Dodgers")

[605141, 518692, 669257, 571970, 606192, 666158, 681546, 621035, 571771]
Away lineup IDs: [592450, 665742, 519317, 650402, 519203, 657077, 669224, 683011, 663757]
Home lineup IDs: [605141, 518692, 669257, 571970, 606192, 666158, 681546, 621035, 571771]
Away pitcher ID: 543037
Home pitcher ID: 808967
Yankees avg runs: 5.50
Dodgers avg runs: 7.06

Yankees win %: 39.7%
Dodgers win %: 60.2%
Unresolved tie %: 0.1%


In [97]:
print("League PA distribution used in simulator:")

for k,v in league_rates.items():
    print(f"{k:10} {v:.3f}")

League PA distribution used in simulator:
out        0.470
strikeout  0.227
single     0.141
walk       0.084
double     0.043
home_run   0.031
triple     0.004


In [89]:
# ============================================================
# CELL 20: inspect CatBoost PA model features
# ============================================================

model_feature_names = pa_model.feature_names_

print("Number of model features:", len(model_feature_names))
print("First 50 features:")
print(model_feature_names[:50])

Number of model features: 11
First 50 features:
['pitcher', 'batter', 'p_throws', 'stand', 'inning', 'outs_when_up', 'balls', 'strikes', 'on_1b', 'on_2b', 'on_3b']


In [90]:
# ============================================================
# CELL 21: build hitter/pitcher feature tables for CatBoost
# ============================================================

# ----------------------------
# Hitter pregame features
# ----------------------------
hitter_pa = (
    pa_df.groupby("batter", as_index=False)
    .agg(
        batter_pa=("pa_class_simple", "size"),
        batter_k_rate=("pa_class_simple", lambda s: (s == "strikeout").mean()),
        batter_bb_rate=("pa_class_simple", lambda s: (s == "walk").mean()),
        batter_1b_rate=("pa_class_simple", lambda s: (s == "single").mean()),
        batter_2b_rate=("pa_class_simple", lambda s: (s == "double").mean()),
        batter_3b_rate=("pa_class_simple", lambda s: (s == "triple").mean()),
        batter_hr_rate=("pa_class_simple", lambda s: (s == "home_run").mean()),
        batter_out_rate=("pa_class_simple", lambda s: (s == "out").mean()),
        stand=("stand", lambda s: s.dropna().mode().iloc[0] if len(s.dropna()) else "R")
    )
)

# ----------------------------
# Pitcher pregame features
# ----------------------------
pitcher_pa = (
    pa_df.groupby("pitcher", as_index=False)
    .agg(
        pitcher_pa=("pa_class_simple", "size"),
        pitcher_k_rate=("pa_class_simple", lambda s: (s == "strikeout").mean()),
        pitcher_bb_rate=("pa_class_simple", lambda s: (s == "walk").mean()),
        pitcher_1b_rate=("pa_class_simple", lambda s: (s == "single").mean()),
        pitcher_2b_rate=("pa_class_simple", lambda s: (s == "double").mean()),
        pitcher_3b_rate=("pa_class_simple", lambda s: (s == "triple").mean()),
        pitcher_hr_rate=("pa_class_simple", lambda s: (s == "home_run").mean()),
        pitcher_out_rate=("pa_class_simple", lambda s: (s == "out").mean()),
        p_throws=("p_throws", lambda s: s.dropna().mode().iloc[0] if len(s.dropna()) else "R")
    )
)

# ----------------------------
# Hitter split features by pitcher handedness
# ----------------------------
hitter_split = (
    pa_df.groupby(["batter", "p_throws"], as_index=False)
    .agg(
        split_pa=("pa_class_simple", "size"),
        split_k_rate=("pa_class_simple", lambda s: (s == "strikeout").mean()),
        split_bb_rate=("pa_class_simple", lambda s: (s == "walk").mean()),
        split_1b_rate=("pa_class_simple", lambda s: (s == "single").mean()),
        split_2b_rate=("pa_class_simple", lambda s: (s == "double").mean()),
        split_3b_rate=("pa_class_simple", lambda s: (s == "triple").mean()),
        split_hr_rate=("pa_class_simple", lambda s: (s == "home_run").mean()),
        split_out_rate=("pa_class_simple", lambda s: (s == "out").mean())
    )
)

# ----------------------------
# Pitcher split features by batter handedness
# ----------------------------
pitcher_split = (
    pa_df.groupby(["pitcher", "stand"], as_index=False)
    .agg(
        p_split_pa=("pa_class_simple", "size"),
        p_split_k_rate=("pa_class_simple", lambda s: (s == "strikeout").mean()),
        p_split_bb_rate=("pa_class_simple", lambda s: (s == "walk").mean()),
        p_split_1b_rate=("pa_class_simple", lambda s: (s == "single").mean()),
        p_split_2b_rate=("pa_class_simple", lambda s: (s == "double").mean()),
        p_split_3b_rate=("pa_class_simple", lambda s: (s == "triple").mean()),
        p_split_hr_rate=("pa_class_simple", lambda s: (s == "home_run").mean()),
        p_split_out_rate=("pa_class_simple", lambda s: (s == "out").mean())
    )
)

print("hitter_pa:", hitter_pa.shape)
print("pitcher_pa:", pitcher_pa.shape)
print("hitter_split:", hitter_split.shape)
print("pitcher_split:", pitcher_split.shape)

hitter_pa: (3243, 10)
pitcher_pa: (2663, 10)
hitter_split: (5541, 10)
pitcher_split: (5149, 10)


In [91]:
# ============================================================
# CELL 22: index lookup tables
# ============================================================

hitter_pa_idx = hitter_pa.set_index("batter")
pitcher_pa_idx = pitcher_pa.set_index("pitcher")
hitter_split_idx = hitter_split.set_index(["batter", "p_throws"])
pitcher_split_idx = pitcher_split.set_index(["pitcher", "stand"])

league_defaults = {
    "batter_pa": 200,
    "pitcher_pa": 250,
    "split_pa": 75,
    "p_split_pa": 75,
    "batter_k_rate": league_rates["strikeout"],
    "batter_bb_rate": league_rates["walk"],
    "batter_1b_rate": league_rates["single"],
    "batter_2b_rate": league_rates["double"],
    "batter_3b_rate": league_rates["triple"],
    "batter_hr_rate": league_rates["home_run"],
    "batter_out_rate": league_rates["out"],
    "pitcher_k_rate": league_rates["strikeout"],
    "pitcher_bb_rate": league_rates["walk"],
    "pitcher_1b_rate": league_rates["single"],
    "pitcher_2b_rate": league_rates["double"],
    "pitcher_3b_rate": league_rates["triple"],
    "pitcher_hr_rate": league_rates["home_run"],
    "pitcher_out_rate": league_rates["out"],
    "split_k_rate": league_rates["strikeout"],
    "split_bb_rate": league_rates["walk"],
    "split_1b_rate": league_rates["single"],
    "split_2b_rate": league_rates["double"],
    "split_3b_rate": league_rates["triple"],
    "split_hr_rate": league_rates["home_run"],
    "split_out_rate": league_rates["out"],
    "p_split_k_rate": league_rates["strikeout"],
    "p_split_bb_rate": league_rates["walk"],
    "p_split_1b_rate": league_rates["single"],
    "p_split_2b_rate": league_rates["double"],
    "p_split_3b_rate": league_rates["triple"],
    "p_split_hr_rate": league_rates["home_run"],
    "p_split_out_rate": league_rates["out"],
    "stand": "R",
    "p_throws": "R",
}

In [92]:
# ============================================================
# CELL 23: build matchup feature row for CatBoost
# ============================================================

def get_row_or_default(indexed_df, key, default_dict):
    if key in indexed_df.index:
        row = indexed_df.loc[key]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return row.to_dict()
    return {}

def build_model_feature_row(batter_id, pitcher_id):
    row = {}

    # ids
    row["batter"] = batter_id
    row["pitcher"] = pitcher_id

    # hitter base
    h = get_row_or_default(hitter_pa_idx, batter_id, league_defaults)
    row.update(h)

    # pitcher base
    p = get_row_or_default(pitcher_pa_idx, pitcher_id, league_defaults)
    row.update(p)

    # handedness defaults
    stand = row.get("stand", "R")
    p_throws = row.get("p_throws", "R")

    # hitter split vs pitcher handedness
    hs = get_row_or_default(hitter_split_idx, (batter_id, p_throws), league_defaults)
    row.update(hs)

    # pitcher split vs batter handedness
    ps = get_row_or_default(pitcher_split_idx, (pitcher_id, stand), league_defaults)
    row.update(ps)

    # basic game-state defaults for pregame neutral PA
    neutral_defaults = {
        "balls": 0,
        "strikes": 0,
        "outs_when_up": 0,
        "on_1b": 0,
        "on_2b": 0,
        "on_3b": 0,
        "inning": 1,
        "bat_score": 0,
        "fld_score": 0,
        "home_team": "UNK",
        "away_team": "UNK",
        "inning_topbot": "Top",
        "if_fielding_alignment": "Standard",
        "of_fielding_alignment": "Standard",
    }

    for k, v in neutral_defaults.items():
        if k not in row:
            row[k] = v

    # fill any remaining missing features with neutral values
    final_row = {}
    for feat in model_feature_names:
        if feat in row:
            final_row[feat] = row[feat]
        else:
            # feature-wise neutral fill
            if feat in ["stand", "p_throws", "inning_topbot", "home_team", "away_team",
                        "if_fielding_alignment", "of_fielding_alignment"]:
                if feat == "stand":
                    final_row[feat] = stand
                elif feat == "p_throws":
                    final_row[feat] = p_throws
                elif feat == "inning_topbot":
                    final_row[feat] = "Top"
                elif feat in ["if_fielding_alignment", "of_fielding_alignment"]:
                    final_row[feat] = "Standard"
                else:
                    final_row[feat] = "UNK"
            else:
                final_row[feat] = 0

    return pd.DataFrame([final_row])

In [93]:
# ============================================================
# CELL 24: CatBoost PA probability wrapper
# ============================================================

def get_catboost_matchup_probs(batter_id, pitcher_id, verbose=False):
    try:
        X = build_model_feature_row(batter_id, pitcher_id)
        probs = pa_model.predict_proba(X)[0]

        # Try to match class order
        if hasattr(pa_model, "classes_") and pa_model.classes_ is not None:
            model_classes = list(pa_model.classes_)
        else:
            model_classes = PA_CLASSES

        prob_map = {cls: prob for cls, prob in zip(model_classes, probs)}

        arr = np.array([prob_map.get(c, 0.0) for c in PA_CLASSES], dtype=float)
        arr = np.clip(arr, 1e-9, None)
        arr = arr / arr.sum()

        return arr

    except Exception as e:
        if verbose:
            print("CatBoost fallback triggered:", e)
        return get_matchup_probs(batter_id, pitcher_id)

In [110]:
# ============================================================
# RUN ENVIRONMENT CALIBRATION
# ============================================================

RUN_ENV_MULTIPLIERS = {
    "out": 1.08,
    "strikeout": 1.03,
    "walk": 0.94,
    "single": 0.93,
    "double": 0.92,
    "triple": 0.90,
    "home_run": 0.88
}

def apply_run_environment_adjustment(probs):
    adjusted = np.array(probs, dtype=float).copy()

    for i, cls in enumerate(PA_CLASSES):
        adjusted[i] *= RUN_ENV_MULTIPLIERS.get(cls, 1.0)

    adjusted = np.clip(adjusted, 1e-9, None)
    adjusted = adjusted / adjusted.sum()
    return adjusted

In [111]:
# ============================================================
# SIMULATE PA WITH CALIBRATION
# ============================================================

USE_CATBOOST_PA_MODEL = True

def simulate_pa(batter_id, pitcher_id):
    if USE_CATBOOST_PA_MODEL:
        probs = get_catboost_matchup_probs(batter_id, pitcher_id)
    else:
        probs = get_matchup_probs(batter_id, pitcher_id)

    probs = apply_run_environment_adjustment(probs)

    outcome = np.random.choice(PA_CLASSES, p=probs)
    return outcome

In [95]:
# ============================================================
# CELL 26: sanity check CatBoost matchup probabilities
# ============================================================

example_batter = away_lineup_ids[0]
example_pitcher = home_pitcher_id

print("Example batter ID:", example_batter)
print("Example pitcher ID:", example_pitcher)

cb_probs = get_catboost_matchup_probs(example_batter, example_pitcher, verbose=True)

print("\nCatBoost matchup probabilities:")
for c, p in zip(PA_CLASSES, cb_probs):
    print(f"{c:10} {p:.4f}")

Example batter ID: 592450
Example pitcher ID: 808967

CatBoost matchup probabilities:
out        0.5961
strikeout  0.0040
walk       0.0025
single     0.2825
double     0.1092
triple     0.0058
home_run   0.0000


In [98]:
way_lineup = [
"Aaron Judge",
"Juan Soto",
"Giancarlo Stanton",
"Gleyber Torres",
"Anthony Rizzo",
"Alex Verdugo",
"Austin Wells",
"Anthony Volpe",
"Trent Grisham"
]

home_lineup = [
    "Mookie Betts",
    "Freddie Freeman",
    "Will Smith",
    "Max Muncy",
    "Teoscar Hernandez",
    "Gavin Lux",
    "James Outman",
    "Chris Taylor",
    "Enrique Hernandez"
]

home_lineup_ids = lineup_to_ids(home_lineup)
print(home_lineup_ids)
validate_lineup_ids(home_lineup_ids, "Home")

away_pitcher_id = get_player_id("Gerrit", "Cole")
home_pitcher_id = get_player_id("Yoshinobu", "Yamamoto")

print("Away lineup IDs:", away_lineup_ids)
print("Home lineup IDs:", home_lineup_ids)
print("Away pitcher ID:", away_pitcher_id)
print("Home pitcher ID:", home_pitcher_id)

sim_result = simulate_matchup(
    away_lineup_ids=away_lineup_ids,
    home_lineup_ids=home_lineup_ids,
    away_pitcher_id=away_pitcher_id,
    home_pitcher_id=home_pitcher_id,
    n_sims=1000
)

print_matchup_summary(sim_result, away_team="Yankees", home_team="Dodgers")

[605141, 518692, 669257, 571970, 606192, 666158, 681546, 621035, 571771]
Away lineup IDs: [592450, 665742, 519317, 650402, 519203, 657077, 669224, 683011, 663757]
Home lineup IDs: [605141, 518692, 669257, 571970, 606192, 666158, 681546, 621035, 571771]
Away pitcher ID: 543037
Home pitcher ID: 808967
Yankees avg runs: 5.50
Dodgers avg runs: 6.86

Yankees win %: 39.1%
Dodgers win %: 60.6%
Unresolved tie %: 0.3%


In [99]:
# ============================================================
# CELL 28: starter leash table
# ============================================================

starter_usage_filtered = starter_usage.copy()

# Keep all starters, but mark more trustworthy ones
starter_usage_filtered["starter_quality_sample"] = np.where(
    starter_usage_filtered["starts"] >= 10,
    "trusted",
    "small_sample"
)

# Build indexed lookup
starter_usage_idx = starter_usage_filtered.set_index("pitcher")

def get_starter_leash(pitcher_id):
    """
    Returns estimated starter leash for a pitcher.
    """
    default_pitch_cap = 95
    default_innings_cap = 5.5

    if pitcher_id not in starter_usage_idx.index:
        return {
            "pitch_cap": default_pitch_cap,
            "innings_cap": default_innings_cap,
            "avg_pitches": default_pitch_cap,
            "avg_innings": default_innings_cap,
            "starts": 0
        }

    row = starter_usage_idx.loc[pitcher_id]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]

    starts = float(row.get("starts", 0))
    avg_pitches = float(row.get("avg_pitches", default_pitch_cap))
    avg_innings = float(row.get("avg_innings_seen", default_innings_cap))

    # Stabilize toward MLB-ish defaults for small samples
    weight = min(starts / 15.0, 1.0)

    pitch_cap = (weight * avg_pitches) + ((1 - weight) * default_pitch_cap)
    innings_cap = (weight * avg_innings) + ((1 - weight) * default_innings_cap)

    # Slight soft upward allowance so starters are not yanked too early
    pitch_cap = min(max(pitch_cap + 5, 75), 110)
    innings_cap = min(max(innings_cap + 0.3, 4.0), 7.5)

    return {
        "pitch_cap": float(pitch_cap),
        "innings_cap": float(innings_cap),
        "avg_pitches": float(avg_pitches),
        "avg_innings": float(avg_innings),
        "starts": int(starts)
    }

print("Starter leash table ready.")
print(get_starter_leash(543037))  # Gerrit Cole example

Starter leash table ready.
{'pitch_cap': 100.9922480620155, 'innings_cap': 6.486046511627907, 'avg_pitches': 95.9922480620155, 'avg_innings': 6.186046511627907, 'starts': 129}


In [100]:
# ============================================================
# CELL 29: reliever role pools
# ============================================================

reliever_usage_work = reliever_usage.copy()

# Only keep relievers with some real sample
reliever_usage_work = reliever_usage_work[reliever_usage_work["relief_apps"] >= 5].copy()

# Build role pools
reliever_usage_idx = reliever_usage_work.set_index("pitcher")

ROLE_ORDER = ["long", "middle", "setup", "closer"]

def get_team_pitcher_pool(team_code, pa_source_df=pa_df):
    """
    Infer a team's pitchers from Statcast appearances.
    Uses pitching_team derived from inning_topbot.
    """
    temp = pa_source_df.copy()
    temp["pitching_team"] = np.where(
        temp["inning_topbot"].astype(str).str.lower().eq("top"),
        temp["home_team"],
        temp["away_team"]
    )

    team_pitchers = temp.loc[temp["pitching_team"] == team_code, "pitcher"].dropna().unique().tolist()
    return team_pitchers

def build_team_bullpen(team_code, starter_id):
    """
    Build a team's bullpen candidates from reliever_usage + team pitcher history.
    """
    team_pitchers = set(get_team_pitcher_pool(team_code))
    bullpen = reliever_usage_work[
        reliever_usage_work["pitcher"].isin(team_pitchers) &
        (reliever_usage_work["pitcher"] != starter_id)
    ].copy()

    if bullpen.empty:
        return bullpen

    # Prefer more-used relievers
    bullpen = bullpen.sort_values(
        ["role_guess", "relief_apps", "avg_relief_pitches"],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    return bullpen

print("Bullpen helper ready.")

Bullpen helper ready.


In [101]:
# ============================================================
# CELL 30: pitcher state + pitch count estimates
# ============================================================

PITCH_ESTIMATES = {
    "out": 3.6,
    "strikeout": 4.4,
    "walk": 5.2,
    "single": 3.5,
    "double": 3.7,
    "triple": 4.0,
    "home_run": 3.8
}

def estimate_pitches_for_pa(outcome):
    base = PITCH_ESTIMATES.get(outcome, 3.8)
    # Add a little randomness
    val = np.random.normal(loc=base, scale=0.8)
    return max(1, int(round(val)))

def init_pitcher_state(pitcher_id, role="starter"):
    leash = get_starter_leash(pitcher_id) if role == "starter" else None

    if role == "starter":
        pitch_cap = leash["pitch_cap"]
        inning_cap = leash["innings_cap"]
    else:
        # relievers usually shorter leash
        if pitcher_id in reliever_usage_idx.index:
            row = reliever_usage_idx.loc[pitcher_id]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            avg_relief_pitches = float(row.get("avg_relief_pitches", 18))
        else:
            avg_relief_pitches = 18

        pitch_cap = min(max(avg_relief_pitches + 5, 15), 35)
        inning_cap = 1.3

    return {
        "pitcher_id": pitcher_id,
        "role": role,
        "pitch_count": 0,
        "batters_faced": 0,
        "innings_started": 0,
        "outs_recorded": 0,
        "runs_allowed": 0,
        "current_inning_runs": 0,
        "pitch_cap": float(pitch_cap),
        "inning_cap": float(inning_cap),
        "active": True
    }

print("Pitcher state helpers ready.")

Pitcher state helpers ready.


In [102]:
# ============================================================
# CELL 31: choose next reliever
# ============================================================

def choose_next_reliever(bullpen_df, used_pitchers, inning, score_diff):
    """
    score_diff = batting_team_runs - fielding_team_runs from the pitching team's perspective:
        positive means pitching team is ahead
        negative means pitching team is behind
    """
    if bullpen_df is None or bullpen_df.empty:
        return None

    available = bullpen_df[~bullpen_df["pitcher"].isin(used_pitchers)].copy()
    if available.empty:
        return None

    # Blowout logic: save good arms in low-leverage spots
    low_leverage = abs(score_diff) >= 6

    if inning >= 9 and not low_leverage:
        pref_roles = ["closer", "setup", "middle", "long"]
    elif inning == 8 and not low_leverage:
        pref_roles = ["setup", "closer", "middle", "long"]
    elif inning <= 5:
        pref_roles = ["long", "middle", "setup", "closer"]
    else:
        pref_roles = ["middle", "setup", "long", "closer"]

    if low_leverage:
        pref_roles = ["long", "middle", "setup", "closer"]

    for role in pref_roles:
        candidates = available[available["role_guess"] == role].copy()
        if not candidates.empty:
            candidates = candidates.sort_values(
                ["relief_apps", "avg_relief_pitches"],
                ascending=[False, False]
            )
            return int(candidates.iloc[0]["pitcher"])

    return int(available.iloc[0]["pitcher"])

In [103]:
# ============================================================
# CELL 32: pitcher removal logic
# ============================================================

def should_remove_pitcher(p_state, inning, half, score_diff, just_allowed_runs=0):
    """
    score_diff is from pitching team's perspective:
    + means pitching team ahead
    - means pitching team behind
    """
    role = p_state["role"]

    innings_completed_est = p_state["outs_recorded"] / 3.0

    # Track inning blow-up
    if just_allowed_runs > 0:
        p_state["current_inning_runs"] += just_allowed_runs

    # Starter rules
    if role == "starter":
        if p_state["pitch_count"] >= p_state["pitch_cap"]:
            return True

        if innings_completed_est >= p_state["inning_cap"] and inning >= 5:
            return True

        # Third-time-ish / late-game hook
        if inning >= 6 and p_state["pitch_count"] >= max(p_state["pitch_cap"] - 10, 75):
            return True

        # Getting shelled
        if p_state["current_inning_runs"] >= 4:
            return True

        if p_state["runs_allowed"] >= 6 and inning <= 5:
            return True

        return False

    # Reliever rules
    else:
        if p_state["pitch_count"] >= p_state["pitch_cap"]:
            return True

        if innings_completed_est >= p_state["inning_cap"] and p_state["pitch_count"] >= 12:
            return True

        # Blow-up rule you wanted
        if p_state["current_inning_runs"] >= 3:
            return True

        # Keep relievers mostly to about one inning
        if p_state["batters_faced"] >= 6 and inning >= 6:
            return True

        return False

In [104]:
# ============================================================
# CELL 33: simulate half inning with pitcher management
# ============================================================

def simulate_half_inning_with_pitching(
    lineup_ids,
    lineup_pos,
    pitching_team_code,
    batting_team_runs,
    pitching_team_runs,
    inning,
    half,
    current_pitcher_state,
    bullpen_df,
    used_pitchers
):
    outs = 0
    runs = 0
    bases = [0, 0, 0]

    # reset inning damage tracker for active pitcher
    current_pitcher_state["current_inning_runs"] = 0

    while outs < 3:
        batter_id = lineup_ids[lineup_pos % 9]
        pitcher_id = current_pitcher_state["pitcher_id"]

        outcome = simulate_pa(batter_id, pitcher_id)
        est_pitches = estimate_pitches_for_pa(outcome)

        current_pitcher_state["pitch_count"] += est_pitches
        current_pitcher_state["batters_faced"] += 1

        bases, new_runs, new_outs = advance_runners(bases, outcome)

        runs += new_runs
        outs += new_outs

        current_pitcher_state["outs_recorded"] += new_outs
        current_pitcher_state["runs_allowed"] += new_runs

        lineup_pos += 1

        score_diff = pitching_team_runs - (batting_team_runs + runs)

        # Mid-inning pitching change if needed
        if outs < 3 and should_remove_pitcher(
            current_pitcher_state,
            inning=inning,
            half=half,
            score_diff=score_diff,
            just_allowed_runs=new_runs
        ):
            next_pitcher_id = choose_next_reliever(
                bullpen_df=bullpen_df,
                used_pitchers=used_pitchers,
                inning=inning,
                score_diff=score_diff
            )

            if next_pitcher_id is not None:
                used_pitchers.add(next_pitcher_id)
                current_pitcher_state = init_pitcher_state(next_pitcher_id, role="reliever")
                current_pitcher_state["current_inning_runs"] = 0

    return runs, lineup_pos, current_pitcher_state, used_pitchers

In [105]:
# ============================================================
# CELL 34: full game with starters + bullpen
# ============================================================

def simulate_game_with_pitching(
    away_lineup_ids,
    home_lineup_ids,
    away_pitcher_id,
    home_pitcher_id,
    away_team_code="AWAY",
    home_team_code="HOME",
    max_extra_innings=6
):
    away_runs = 0
    home_runs = 0

    away_lineup_pos = 0
    home_lineup_pos = 0

    # Build team bullpens
    away_bullpen = build_team_bullpen(away_team_code, away_pitcher_id)
    home_bullpen = build_team_bullpen(home_team_code, home_pitcher_id)

    used_away_pitchers = {away_pitcher_id}
    used_home_pitchers = {home_pitcher_id}

    away_pitcher_state = init_pitcher_state(away_pitcher_id, role="starter")
    home_pitcher_state = init_pitcher_state(home_pitcher_id, role="starter")

    # 9 innings
    for inning in range(1, 10):
        # Top: away bats, home pitches
        top_runs, away_lineup_pos, home_pitcher_state, used_home_pitchers = simulate_half_inning_with_pitching(
            lineup_ids=away_lineup_ids,
            lineup_pos=away_lineup_pos,
            pitching_team_code=home_team_code,
            batting_team_runs=away_runs,
            pitching_team_runs=home_runs,
            inning=inning,
            half="top",
            current_pitcher_state=home_pitcher_state,
            bullpen_df=home_bullpen,
            used_pitchers=used_home_pitchers
        )
        away_runs += top_runs

        # Bottom: home bats, away pitches
        bot_runs, home_lineup_pos, away_pitcher_state, used_away_pitchers = simulate_half_inning_with_pitching(
            lineup_ids=home_lineup_ids,
            lineup_pos=home_lineup_pos,
            pitching_team_code=away_team_code,
            batting_team_runs=home_runs,
            pitching_team_runs=away_runs,
            inning=inning,
            half="bottom",
            current_pitcher_state=away_pitcher_state,
            bullpen_df=away_bullpen,
            used_pitchers=used_away_pitchers
        )
        home_runs += bot_runs

        # End-of-inning starter hook if leash reached
        if should_remove_pitcher(
            home_pitcher_state,
            inning=inning,
            half="end_top",
            score_diff=home_runs - away_runs,
            just_allowed_runs=0
        ):
            next_pitcher_id = choose_next_reliever(
                bullpen_df=home_bullpen,
                used_pitchers=used_home_pitchers,
                inning=inning + 1,
                score_diff=home_runs - away_runs
            )
            if next_pitcher_id is not None and next_pitcher_id != home_pitcher_state["pitcher_id"]:
                used_home_pitchers.add(next_pitcher_id)
                home_pitcher_state = init_pitcher_state(next_pitcher_id, role="reliever")

        if should_remove_pitcher(
            away_pitcher_state,
            inning=inning,
            half="end_bottom",
            score_diff=away_runs - home_runs,
            just_allowed_runs=0
        ):
            next_pitcher_id = choose_next_reliever(
                bullpen_df=away_bullpen,
                used_pitchers=used_away_pitchers,
                inning=inning + 1,
                score_diff=away_runs - home_runs
            )
            if next_pitcher_id is not None and next_pitcher_id != away_pitcher_state["pitcher_id"]:
                used_away_pitchers.add(next_pitcher_id)
                away_pitcher_state = init_pitcher_state(next_pitcher_id, role="reliever")

    # Extras
    extras_used = 0
    inning = 10
    while away_runs == home_runs and extras_used < max_extra_innings:
        top_runs, away_lineup_pos, home_pitcher_state, used_home_pitchers = simulate_half_inning_with_pitching(
            lineup_ids=away_lineup_ids,
            lineup_pos=away_lineup_pos,
            pitching_team_code=home_team_code,
            batting_team_runs=away_runs,
            pitching_team_runs=home_runs,
            inning=inning,
            half="top",
            current_pitcher_state=home_pitcher_state,
            bullpen_df=home_bullpen,
            used_pitchers=used_home_pitchers
        )
        away_runs += top_runs

        bot_runs, home_lineup_pos, away_pitcher_state, used_away_pitchers = simulate_half_inning_with_pitching(
            lineup_ids=home_lineup_ids,
            lineup_pos=home_lineup_pos,
            pitching_team_code=away_team_code,
            batting_team_runs=home_runs,
            pitching_team_runs=away_runs,
            inning=inning,
            half="bottom",
            current_pitcher_state=away_pitcher_state,
            bullpen_df=away_bullpen,
            used_pitchers=used_away_pitchers
        )
        home_runs += bot_runs

        inning += 1
        extras_used += 1

    return {
        "away_runs": away_runs,
        "home_runs": home_runs,
        "used_away_pitchers": list(used_away_pitchers),
        "used_home_pitchers": list(used_home_pitchers)
    }

In [106]:
# ============================================================
# CELL 35: Monte Carlo wrapper with pitching changes
# ============================================================

def simulate_matchup_with_pitching(
    away_lineup_ids,
    home_lineup_ids,
    away_pitcher_id,
    home_pitcher_id,
    away_team_code,
    home_team_code,
    n_sims=1000
):
    results = []

    for _ in range(n_sims):
        g = simulate_game_with_pitching(
            away_lineup_ids=away_lineup_ids,
            home_lineup_ids=home_lineup_ids,
            away_pitcher_id=away_pitcher_id,
            home_pitcher_id=home_pitcher_id,
            away_team_code=away_team_code,
            home_team_code=home_team_code
        )
        results.append(g)

    results_df = pd.DataFrame(results)

    away_avg = results_df["away_runs"].mean()
    home_avg = results_df["home_runs"].mean()

    away_wins = (results_df["away_runs"] > results_df["home_runs"]).mean()
    home_wins = (results_df["home_runs"] > results_df["away_runs"]).mean()
    unresolved_ties = (results_df["home_runs"] == results_df["away_runs"]).mean()

    return {
        "results_df": results_df,
        "away_avg_runs": away_avg,
        "home_avg_runs": home_avg,
        "away_win_pct": away_wins,
        "home_win_pct": home_wins,
        "unresolved_tie_pct": unresolved_ties
    }

In [108]:
# ============================================================
# SCORE DISTRIBUTION SUMMARY
# ============================================================

def summarize_score_distribution(sim_result):

    df = sim_result["results_df"].copy()

    df["score_pair"] = (
        df["away_runs"].astype(str) + "-" + df["home_runs"].astype(str)
    )

    most_common = df["score_pair"].value_counts().head(10)

    print("\nTop 10 most common final scores:")
    print(most_common)

    print("\nMedian score:")
    print("Away:", df["away_runs"].median())
    print("Home:", df["home_runs"].median())

    print("\nMost common score:", most_common.index[0])

In [112]:
# ============================================================
# CELL 36: test matchup with pitching changes
# ============================================================

sim_result_pitching = simulate_matchup_with_pitching(
    away_lineup_ids=away_lineup_ids,
    home_lineup_ids=home_lineup_ids,
    away_pitcher_id=away_pitcher_id,
    home_pitcher_id=home_pitcher_id,
    away_team_code="NYY",
    home_team_code="LAD",
    n_sims=300
)

print_matchup_summary(sim_result_pitching, away_team="Yankees", home_team="Dodgers")
summarize_score_distribution(sim_result_pitching)

Yankees avg runs: 4.67
Dodgers avg runs: 4.47

Yankees win %: 49.7%
Dodgers win %: 50.0%
Unresolved tie %: 0.3%

Top 10 most common final scores:
score_pair
2-6    11
2-4     9
1-3     9
3-4     8
2-3     7
0-2     7
5-4     7
4-3     6
6-3     6
2-5     5
Name: count, dtype: int64

Median score:
Away: 4.0
Home: 4.0

Most common score: 2-6


In [113]:
# ============================================================
# CELL A: imports + MLB game feed helpers
# ============================================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

from pybaseball import cache

# optional but recommended for repeated pulls
cache.enable()

BASE_MLB_API = "https://statsapi.mlb.com/api"

TEAM_ABBR_TO_ID = {
    "ARI": 109, "ATL": 144, "BAL": 110, "BOS": 111, "CHC": 112, "CWS": 145,
    "CIN": 113, "CLE": 114, "COL": 115, "DET": 116, "HOU": 117, "KC": 118,
    "LAA": 108, "LAD": 119, "MIA": 146, "MIL": 158, "MIN": 142, "NYM": 121,
    "NYY": 147, "ATH": 133, "PHI": 143, "PIT": 134, "SD": 135, "SEA": 136,
    "SF": 137, "STL": 138, "TB": 139, "TEX": 140, "TOR": 141, "WSH": 120
}

def mlb_get(path, params=None, timeout=20):
    url = f"{BASE_MLB_API}{path}"
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def safe_name(player_obj):
    if not player_obj:
        return None
    return player_obj.get("fullName") or player_obj.get("name")

def normalize_batting_order(order_list):
    """
    MLB feed often stores batting order entries like '100', '200', ... '900'.
    Sort and convert to player IDs.
    """
    if not order_list:
        return []

    cleaned = []
    for x in order_list:
        sx = str(x)
        try:
            cleaned.append((int(sx), sx))
        except ValueError:
            continue

    cleaned = sorted(cleaned, key=lambda t: t[0])

    # keep first 9 spots only
    player_ids = []
    for _, sx in cleaned[:9]:
        # battingOrder items are roster slot codes; actual player IDs come from team["players"]
        # handled elsewhere, so this helper is only for ordering
        player_ids.append(sx)
    return player_ids

In [124]:
# ============================================================
# FIXED: today's schedule / matchup list
# ============================================================

def get_daily_games(game_date=None):
    """
    Returns a DataFrame of MLB games for a date.
    """
    if game_date is None:
        game_date = datetime.now().strftime("%Y-%m-%d")

    data = mlb_get(
        "/v1/schedule",
        params={
            "sportId": 1,
            "date": game_date,
            "hydrate": "probablePitcher,linescore"
        }
    )

    rows = []
    for date_block in data.get("dates", []):
        for g in date_block.get("games", []):
            away = g.get("teams", {}).get("away", {})
            home = g.get("teams", {}).get("home", {})
            status = g.get("status", {})

            away_team = away.get("team", {})
            home_team = home.get("team", {})

            away_id = away_team.get("id")
            home_id = home_team.get("id")

            away_name = away_team.get("name")
            home_name = home_team.get("name")

            # fallback to manual mapping if abbreviation missing
            away_abbr = away_team.get("abbreviation")
            home_abbr = home_team.get("abbreviation")

            if away_abbr is None:
                away_abbr = next((k for k, v in TEAM_ABBR_TO_ID.items() if v == away_id), away_name)
            if home_abbr is None:
                home_abbr = next((k for k, v in TEAM_ABBR_TO_ID.items() if v == home_id), home_name)

            rows.append({
                "gamePk": g.get("gamePk"),
                "gameDate": g.get("gameDate"),
                "status": status.get("detailedState"),
                "away_name": away_name,
                "away_abbr": away_abbr,
                "away_id": away_id,
                "away_probable_pitcher": safe_name(away.get("probablePitcher")),
                "home_name": home_name,
                "home_abbr": home_abbr,
                "home_id": home_id,
                "home_probable_pitcher": safe_name(home.get("probablePitcher")),
            })

    games_df = pd.DataFrame(rows)

    if not games_df.empty:
        games_df["label"] = (
            games_df["away_abbr"].astype(str) + " @ " +
            games_df["home_abbr"].astype(str) + " | " +
            games_df["status"].fillna("Unknown").astype(str) + " | " +
            games_df["gameDate"].astype(str)
        )

    return games_df

In [115]:
# ============================================================
# CELL B: today's schedule / matchup list
# ============================================================

def get_daily_games(game_date=None):
    """
    Returns a DataFrame of MLB games for a date.
    """
    if game_date is None:
        game_date = datetime.now().strftime("%Y-%m-%d")

    data = mlb_get(
        "/v1/schedule",
        params={
            "sportId": 1,
            "date": game_date,
            "hydrate": "probablePitcher,linescore"
        }
    )

    rows = []
    for date_block in data.get("dates", []):
        for g in date_block.get("games", []):
            away = g.get("teams", {}).get("away", {})
            home = g.get("teams", {}).get("home", {})
            status = g.get("status", {})

            rows.append({
                "gamePk": g.get("gamePk"),
                "gameDate": g.get("gameDate"),
                "status": status.get("detailedState"),
                "away_name": away.get("team", {}).get("name"),
                "away_abbr": away.get("team", {}).get("abbreviation"),
                "away_id": away.get("team", {}).get("id"),
                "away_probable_pitcher": safe_name(away.get("probablePitcher")),
                "home_name": home.get("team", {}).get("name"),
                "home_abbr": home.get("team", {}).get("abbreviation"),
                "home_id": home.get("team", {}).get("id"),
                "home_probable_pitcher": safe_name(home.get("probablePitcher")),
            })

    games_df = pd.DataFrame(rows)

    if not games_df.empty:
        games_df["label"] = (
            games_df["away_abbr"] + " @ " + games_df["home_abbr"]
            + " | " + games_df["status"].fillna("Unknown")
            + " | " + games_df["gameDate"].astype(str)
        )

    return games_df

In [116]:
# ============================================================
# CELL C: active rosters for a team on a date
# ============================================================

def get_team_active_roster(team_id, game_date=None):
    """
    Returns active roster for a team.
    """
    if game_date is None:
        game_date = datetime.now().strftime("%Y-%m-%d")

    # This endpoint pattern is commonly used with the MLB API family.
    data = mlb_get(
        f"/v1/teams/{team_id}/roster",
        params={
            "rosterType": "active",
            "date": game_date
        }
    )

    rows = []
    for p in data.get("roster", []):
        person = p.get("person", {})
        pos = p.get("position", {})
        rows.append({
            "player_id": person.get("id"),
            "player_name": person.get("fullName"),
            "position_code": pos.get("abbreviation"),
            "position_type": pos.get("type"),
            "status_code": p.get("status", {}).get("code"),
            "status_desc": p.get("status", {}).get("description"),
        })

    return pd.DataFrame(rows)

In [117]:
# ============================================================
# CELL D: pull game feed, confirmed lineups, probable/current pitchers, live state
# ============================================================

def _extract_team_lineup_from_boxscore(team_box):
    """
    Returns confirmed lineup if posted, else best available batting order from the feed.
    """
    players = team_box.get("players", {})
    batting_order = team_box.get("battingOrder", [])
    batting_order = normalize_batting_order(batting_order)

    lineup_rows = []

    # confirmed batting order path
    if batting_order and players:
        for slot_code in batting_order:
            # battingOrder is order codes, but players dict contains a 'battingOrder' field
            matched = None
            for player_key, pobj in players.items():
                if str(pobj.get("battingOrder", "")) == str(slot_code):
                    matched = pobj
                    break

            if matched is None:
                continue

            person = matched.get("person", {})
            position = matched.get("position", {})

            lineup_rows.append({
                "player_id": person.get("id"),
                "player_name": person.get("fullName"),
                "bat_order": matched.get("battingOrder"),
                "position": position.get("abbreviation"),
                "bat_side": matched.get("batSide", {}).get("code"),
                "pitch_hand": matched.get("pitchHand", {}).get("code")
            })

    lineup_df = pd.DataFrame(lineup_rows)

    if not lineup_df.empty:
        lineup_df = lineup_df.sort_values("bat_order").reset_index(drop=True)

    return lineup_df

def _extract_current_pitcher_from_boxscore(team_box):
    for player_key, pobj in team_box.get("players", {}).items():
        if pobj.get("position", {}).get("abbreviation") == "P" and pobj.get("stats", {}).get("pitching") is not None:
            # not guaranteed to be current pitcher, but useful fallback
            if pobj.get("gameStatus", {}).get("isCurrentPitcher"):
                return {
                    "player_id": pobj.get("person", {}).get("id"),
                    "player_name": pobj.get("person", {}).get("fullName")
                }
    return None

def get_game_feed(game_pk):
    return mlb_get(f"/v1.1/game/{game_pk}/feed/live")

def get_game_snapshot(game_pk):
    """
    Pulls one game's feed and returns:
    - away/home lineups
    - active players
    - probable pitchers
    - live inning / outs / bases / score
    """
    feed = get_game_feed(game_pk)

    game_data = feed.get("gameData", {})
    live_data = feed.get("liveData", {})
    boxscore = live_data.get("boxscore", {})
    linescore = live_data.get("linescore", {})
    plays = live_data.get("plays", {})

    away_team = game_data.get("teams", {}).get("away", {})
    home_team = game_data.get("teams", {}).get("home", {})

    away_box = boxscore.get("teams", {}).get("away", {})
    home_box = boxscore.get("teams", {}).get("home", {})

    away_lineup = _extract_team_lineup_from_boxscore(away_box)
    home_lineup = _extract_team_lineup_from_boxscore(home_box)

    away_probable = safe_name(away_team.get("probablePitcher"))
    home_probable = safe_name(home_team.get("probablePitcher"))

    offense = linescore.get("offense", {})
    defense = linescore.get("defense", {})
    current_play = plays.get("currentPlay", {})

    snapshot = {
        "gamePk": game_pk,
        "away_name": away_team.get("name"),
        "away_abbr": away_team.get("abbreviation"),
        "away_id": away_team.get("id"),
        "home_name": home_team.get("name"),
        "home_abbr": home_team.get("abbreviation"),
        "home_id": home_team.get("id"),
        "away_probable_pitcher": away_probable,
        "home_probable_pitcher": home_probable,
        "away_lineup": away_lineup,
        "home_lineup": home_lineup,
        "status": game_data.get("status", {}).get("detailedState"),
        "inning": linescore.get("currentInning"),
        "inning_half": linescore.get("inningHalf"),
        "outs": linescore.get("outs"),
        "away_score": linescore.get("teams", {}).get("away", {}).get("runs"),
        "home_score": linescore.get("teams", {}).get("home", {}).get("runs"),
        "runner_on_1b": offense.get("first", {}).get("id") if offense.get("first") else None,
        "runner_on_2b": offense.get("second", {}).get("id") if offense.get("second") else None,
        "runner_on_3b": offense.get("third", {}).get("id") if offense.get("third") else None,
        "offense_team_abbr": offense.get("team", {}).get("abbreviation") if offense.get("team") else None,
        "defense_team_abbr": defense.get("team", {}).get("abbreviation") if defense.get("team") else None,
        "current_batter": safe_name(offense.get("batter")),
        "current_pitcher": safe_name(defense.get("pitcher")),
        "feed_json": feed
    }

    return snapshot

In [121]:
# ============================================================
# CELL E: lineup resolution logic
# ============================================================

def build_projected_lineup_from_roster(roster_df):
    """
    Very simple fallback:
    - prefer non-pitchers
    - take first 9
    You can later replace this with your projection logic.
    """
    if roster_df.empty:
        return pd.DataFrame(columns=["player_id", "player_name", "position_code"])

    hitters = roster_df[
        ~roster_df["position_type"].astype(str).str.contains("Pitcher", case=False, na=False)
    ].copy()

    hitters = hitters.head(9).reset_index(drop=True)
    hitters["bat_order"] = [(i + 1) * 100 for i in range(len(hitters))]

    return hitters[["player_id", "player_name", "position_code", "bat_order"]]

def resolve_game_lineups(snapshot, game_date=None):
    """
    Use confirmed lineup if available; otherwise use active roster fallback.
    """
    away_lineup = snapshot["away_lineup"]
    home_lineup = snapshot["home_lineup"]

    away_source = "confirmed"
    home_source = "confirmed"

    if away_lineup.empty:
        away_roster = get_team_active_roster(snapshot["away_id"], game_date=game_date)
        away_lineup = build_projected_lineup_from_roster(away_roster)
        away_source = "projected_from_active_roster"

    if home_lineup.empty:
        home_roster = get_team_active_roster(snapshot["home_id"], game_date=game_date)
        home_lineup = build_projected_lineup_from_roster(home_roster)
        home_source = "projected_from_active_roster"

    return {
        "away_lineup": away_lineup,
        "home_lineup": home_lineup,
        "away_source": away_source,
        "home_source": home_source
    }

In [122]:
# ============================================================
# CELL F: dropdown UI for today's games
# ============================================================

def launch_game_selector(game_date=None):
    games_df = get_daily_games(game_date=game_date)

    if games_df.empty:
        print("No MLB games found for that date.")
        return None

    dropdown = widgets.Dropdown(
        options=[(row["label"], int(row["gamePk"])) for _, row in games_df.iterrows()],
        description="Game:",
        layout=widgets.Layout(width="900px")
    )

    refresh_btn = widgets.Button(description="Load Game", button_style="primary")
    output = widgets.Output()

    def on_refresh_clicked(_):
        with output:
            clear_output()

            game_pk = dropdown.value
            selected = games_df.loc[games_df["gamePk"] == game_pk].iloc[0]

            print(f"Selected: {selected['away_abbr']} @ {selected['home_abbr']}")
            print(f"Status: {selected['status']}")
            print(f"GamePk: {selected['gamePk']}")
            print(f"Away probable: {selected['away_probable_pitcher']}")
            print(f"Home probable: {selected['home_probable_pitcher']}")
            print()

            snapshot = get_game_snapshot(game_pk)
            lineup_info = resolve_game_lineups(snapshot, game_date=game_date)

            print("Away lineup source:", lineup_info["away_source"])
            display(lineup_info["away_lineup"])

            print("Home lineup source:", lineup_info["home_source"])
            display(lineup_info["home_lineup"])

            print("Live state:")
            print({
                "inning": snapshot["inning"],
                "inning_half": snapshot["inning_half"],
                "outs": snapshot["outs"],
                "away_score": snapshot["away_score"],
                "home_score": snapshot["home_score"],
                "runner_on_1b": snapshot["runner_on_1b"],
                "runner_on_2b": snapshot["runner_on_2b"],
                "runner_on_3b": snapshot["runner_on_3b"],
                "current_batter": snapshot["current_batter"],
                "current_pitcher": snapshot["current_pitcher"],
            })

    refresh_btn.on_click(on_refresh_clicked)
    display(widgets.VBox([dropdown, refresh_btn, output]))

    return games_df

In [125]:
# ============================================================
# CELL G: run this to pick today's game
# ============================================================

games_today = launch_game_selector()
games_today

,gamePk,gameDate,status,away_name,away_abbr,away_id,away_probable_pitcher,home_name,home_abbr,home_id,home_probable_pitcher,label
0,831625,2026-03-10T17:05:00Z,Final,Detroit Tigers,DET,116,Ty Madden,Boston Red Sox,BOS,111,Sonny Gray,DET @ BOS | Final | 2026-03-10T17:05:00Z
1,831617,2026-03-10T17:05:00Z,Final,Baltimore Orioles,BAL,110,Levi Wells,Houston Astros,HOU,117,Lance McCullers Jr.,BAL @ HOU | Final | 2026-03-10T17:05:00Z
2,831618,2026-03-10T17:05:00Z,Final,Minnesota Twins,MIN,142,Joe Ryan,Tampa Bay Rays,TB,139,Steven Matz,MIN @ TB | Final | 2026-03-10T17:05:00Z
3,831573,2026-03-10T17:05:00Z,Final,New York Yankees,NYY,147,Luis Gil,Philadelphia Phillies,PHI,143,Tanner Banks,NYY @ PHI | Final | 2026-03-10T17:05:00Z
4,831513,2026-03-10T17:07:00Z,Final,Atlanta Braves,ATL,144,JR Ritchie,Toronto Blue Jays,TOR,141,Dylan Cease,ATL @ TOR | Final | 2026-03-10T17:07:00Z
5,831473,2026-03-10T17:10:00Z,Final,St. Louis Cardinals,STL,138,Jared Shuster,New York Mets,NYM,121,David Peterson,STL @ NYM | Final | 2026-03-10T17:10:00Z
6,831471,2026-03-10T17:10:00Z,Final,Washington Nationals,WSH,120,Foster Griffin,Miami Marlins,MIA,146,Eury Pérez,WSH @ MIA | Final | 2026-03-10T17:10:00Z
7,831956,2026-03-10T20:05:00Z,In Progress,Arizona Diamondbacks,ARI,109,Brandon Pfaadt,Los Angeles Dodgers,LAD,119,Tyler Glasnow,ARI @ LAD | In Progress | 2026-03-10T20:05:00Z
8,831758,2026-03-10T20:05:00Z,In Progress,San Francisco Giants,SF,137,Carson Seymour,Cleveland Guardians,CLE,114,Tanner Bibee,SF @ CLE | In Progress | 2026-03-10T20:05:00Z
9,831920,2026-03-10T20:05:00Z,In Progress,Chicago Cubs,CHC,112,Cade Horton,Texas Rangers,TEX,140,Jacob deGrom,CHC @ TEX | In Progress | 2026-03-10T20:05:00Z


In [126]:
# ============================================================
# CELL I: selected-game -> simulator helpers
# ============================================================

def probable_pitcher_id_from_name(name):
    if name is None or str(name).strip() == "":
        return None
    return get_player_id_from_full_name(name)

def extract_feed_ids(snapshot):
    """
    Pull useful IDs directly from feed JSON.
    """
    feed = snapshot["feed_json"]
    live_data = feed.get("liveData", {})
    linescore = live_data.get("linescore", {})
    offense = linescore.get("offense", {}) or {}
    defense = linescore.get("defense", {}) or {}

    current_batter_id = offense.get("batter", {}).get("id") if offense.get("batter") else None
    current_pitcher_id = defense.get("pitcher", {}).get("id") if defense.get("pitcher") else None

    return {
        "current_batter_id": current_batter_id,
        "current_pitcher_id": current_pitcher_id
    }

def lineup_df_to_ids(lineup_df):
    if lineup_df.empty:
        return []
    if "player_id" not in lineup_df.columns:
        raise ValueError("lineup_df must contain player_id")
    return lineup_df["player_id"].dropna().astype(int).tolist()

def count_team_pas_from_feed(feed_json, away_id, home_id):
    """
    Count completed plate appearances by team from play-by-play.
    Used to estimate next lineup spot due up.
    """
    all_plays = feed_json.get("liveData", {}).get("plays", {}).get("allPlays", [])
    away_pa = 0
    home_pa = 0

    for play in all_plays:
        about = play.get("about", {})
        is_top = about.get("isTopInning")

        # each play in allPlays is one completed PA
        if is_top is True:
            away_pa += 1
        elif is_top is False:
            home_pa += 1

    return away_pa, home_pa

def next_lineup_positions_from_feed(snapshot, away_lineup_ids, home_lineup_ids):
    away_pa, home_pa = count_team_pas_from_feed(
        snapshot["feed_json"],
        snapshot["away_id"],
        snapshot["home_id"]
    )

    away_pos = away_pa % max(len(away_lineup_ids), 1)
    home_pos = home_pa % max(len(home_lineup_ids), 1)

    return away_pos, home_pos

In [127]:
# ============================================================
# CELL J: normalize live state to next half-inning start
# ============================================================

def normalize_next_half_inning_state(snapshot):
    """
    We want inning-by-inning updates, so this converts the feed into:
    - start_inning
    - start_half
    - current score
    for the NEXT half-inning forecast point.

    If the game is between halves (outs == 3), this moves to the next half.
    If not, it uses the currently listed half as an approximation for the next update point.
    """
    inning = snapshot.get("inning") or 1
    half = str(snapshot.get("inning_half") or "Top").lower()
    outs = snapshot.get("outs")
    away_score = int(snapshot.get("away_score") or 0)
    home_score = int(snapshot.get("home_score") or 0)

    # default assumption
    start_inning = inning
    start_half = half

    # if between halves, advance to next half
    if outs == 3:
        if half == "top":
            start_half = "bottom"
            start_inning = inning
        elif half == "bottom":
            start_half = "top"
            start_inning = inning + 1

    return {
        "start_inning": int(start_inning),
        "start_half": start_half,
        "away_score": away_score,
        "home_score": home_score
    }

In [128]:
# ============================================================
# CELL K: pitcher resolution for pregame + live
# ============================================================

def get_boxscore_pitcher_stats_for_team(snapshot, side="away"):
    """
    side in {'away','home'}
    Returns dict: pitcher_id -> pitcher stats from boxscore if present
    """
    feed = snapshot["feed_json"]
    team_box = (
        feed.get("liveData", {})
        .get("boxscore", {})
        .get("teams", {})
        .get(side, {})
    )

    out = {}

    for _, pobj in team_box.get("players", {}).items():
        person = pobj.get("person", {})
        pid = person.get("id")
        if pid is None:
            continue

        pstats = pobj.get("stats", {}).get("pitching")
        if pstats is None:
            continue

        out[int(pid)] = {
            "player_name": person.get("fullName"),
            "numberOfPitches": pstats.get("numberOfPitches", 0),
            "outs": pstats.get("outs", 0),
            "runs": pstats.get("runs", 0),
            "isCurrentPitcher": pobj.get("gameStatus", {}).get("isCurrentPitcher", False)
        }

    return out

def get_current_or_fallback_pitcher_id(snapshot, side="away"):
    """
    For live simulation:
    - if team currently has a pitcher flagged in boxscore, use that
    - else fall back to probable starter
    """
    box = get_boxscore_pitcher_stats_for_team(snapshot, side=side)

    for pid, info in box.items():
        if info.get("isCurrentPitcher", False):
            return int(pid)

    # fallback to probable starter by side
    if side == "away":
        return probable_pitcher_id_from_name(snapshot.get("away_probable_pitcher"))
    else:
        return probable_pitcher_id_from_name(snapshot.get("home_probable_pitcher"))

def get_used_pitchers_from_snapshot(snapshot, side="away"):
    box = get_boxscore_pitcher_stats_for_team(snapshot, side=side)
    return set(box.keys())

In [129]:
# ============================================================
# CELL L: live pitcher-state initialization
# ============================================================

def init_live_pitcher_state(snapshot, pitcher_id, side="away", role_hint=None):
    """
    side = team of the pitcher ('away' or 'home')
    Uses live boxscore pitch totals when available.
    """
    if pitcher_id is None:
        return init_pitcher_state(-1, role="starter")

    box = get_boxscore_pitcher_stats_for_team(snapshot, side=side)
    info = box.get(int(pitcher_id), {})

    # decide starter vs reliever
    probable_id = probable_pitcher_id_from_name(
        snapshot.get("away_probable_pitcher") if side == "away" else snapshot.get("home_probable_pitcher")
    )

    if role_hint is not None:
        role = role_hint
    else:
        role = "starter" if probable_id == pitcher_id else "reliever"

    p_state = init_pitcher_state(pitcher_id, role=role)

    p_state["pitch_count"] = int(info.get("numberOfPitches", 0) or 0)
    p_state["outs_recorded"] = int(info.get("outs", 0) or 0)
    p_state["runs_allowed"] = int(info.get("runs", 0) or 0)

    return p_state

In [130]:
# ============================================================
# CELL M: simulate remaining game from next half-inning state
# ============================================================

def simulate_remaining_game_from_state(
    away_lineup_ids,
    home_lineup_ids,
    away_team_code,
    home_team_code,
    away_pitcher_state,
    home_pitcher_state,
    away_used_pitchers,
    home_used_pitchers,
    away_score_start,
    home_score_start,
    start_inning,
    start_half,
    away_lineup_pos,
    home_lineup_pos,
    max_extra_innings=6
):
    away_runs = int(away_score_start)
    home_runs = int(home_score_start)

    away_bullpen = build_team_bullpen(away_team_code, away_pitcher_state["pitcher_id"])
    home_bullpen = build_team_bullpen(home_team_code, home_pitcher_state["pitcher_id"])

    inning = int(start_inning)
    half = str(start_half).lower()

    def play_top_of_inning(inning_num):
        nonlocal away_runs, away_lineup_pos, home_pitcher_state, home_used_pitchers
        top_runs, away_lineup_pos, home_pitcher_state, home_used_pitchers = simulate_half_inning_with_pitching(
            lineup_ids=away_lineup_ids,
            lineup_pos=away_lineup_pos,
            pitching_team_code=home_team_code,
            batting_team_runs=away_runs,
            pitching_team_runs=home_runs,
            inning=inning_num,
            half="top",
            current_pitcher_state=home_pitcher_state,
            bullpen_df=home_bullpen,
            used_pitchers=home_used_pitchers
        )
        away_runs += top_runs

    def play_bottom_of_inning(inning_num):
        nonlocal home_runs, home_lineup_pos, away_pitcher_state, away_used_pitchers
        bot_runs, home_lineup_pos, away_pitcher_state, away_used_pitchers = simulate_half_inning_with_pitching(
            lineup_ids=home_lineup_ids,
            lineup_pos=home_lineup_pos,
            pitching_team_code=away_team_code,
            batting_team_runs=home_runs,
            pitching_team_runs=away_runs,
            inning=inning_num,
            half="bottom",
            current_pitcher_state=away_pitcher_state,
            bullpen_df=away_bullpen,
            used_pitchers=away_used_pitchers
        )
        home_runs += bot_runs

    # start from the normalized next half-inning point
    if half == "top":
        play_top_of_inning(inning)
        play_bottom_of_inning(inning)
        inning += 1
    elif half == "bottom":
        play_bottom_of_inning(inning)
        inning += 1

    # remaining regulation innings
    while inning <= 9:
        play_top_of_inning(inning)
        play_bottom_of_inning(inning)
        inning += 1

    # extras if tied
    extras_used = 0
    while away_runs == home_runs and extras_used < max_extra_innings:
        play_top_of_inning(inning)
        play_bottom_of_inning(inning)
        inning += 1
        extras_used += 1

    return {
        "away_runs": away_runs,
        "home_runs": home_runs,
        "used_away_pitchers": list(away_used_pitchers),
        "used_home_pitchers": list(home_used_pitchers)
    }

In [131]:
# ============================================================
# CELL N: pregame forecast + live updated final forecast
# ============================================================

def run_pregame_forecast_from_snapshot(snapshot, lineup_info, n_sims=200):
    away_lineup_ids = lineup_df_to_ids(lineup_info["away_lineup"])
    home_lineup_ids = lineup_df_to_ids(lineup_info["home_lineup"])

    validate_lineup_ids(away_lineup_ids, "Away")
    validate_lineup_ids(home_lineup_ids, "Home")

    away_pitcher_id = probable_pitcher_id_from_name(snapshot.get("away_probable_pitcher"))
    home_pitcher_id = probable_pitcher_id_from_name(snapshot.get("home_probable_pitcher"))

    if away_pitcher_id is None or home_pitcher_id is None:
        raise ValueError("Could not resolve one or both probable starters.")

    result = simulate_matchup_with_pitching(
        away_lineup_ids=away_lineup_ids,
        home_lineup_ids=home_lineup_ids,
        away_pitcher_id=away_pitcher_id,
        home_pitcher_id=home_pitcher_id,
        away_team_code=snapshot["away_abbr"],
        home_team_code=snapshot["home_abbr"],
        n_sims=n_sims
    )

    return result

def run_live_forecast_from_snapshot(snapshot, lineup_info, n_sims=200):
    """
    Live forecast from the next half-inning start.
    Designed for inning-by-inning updates.
    """
    away_lineup_ids = lineup_df_to_ids(lineup_info["away_lineup"])
    home_lineup_ids = lineup_df_to_ids(lineup_info["home_lineup"])

    validate_lineup_ids(away_lineup_ids, "Away")
    validate_lineup_ids(home_lineup_ids, "Home")

    norm = normalize_next_half_inning_state(snapshot)

    away_lineup_pos, home_lineup_pos = next_lineup_positions_from_feed(
        snapshot,
        away_lineup_ids,
        home_lineup_ids
    )

    away_pitcher_id_live = get_current_or_fallback_pitcher_id(snapshot, side="away")
    home_pitcher_id_live = get_current_or_fallback_pitcher_id(snapshot, side="home")

    away_pitcher_state_live = init_live_pitcher_state(snapshot, away_pitcher_id_live, side="away")
    home_pitcher_state_live = init_live_pitcher_state(snapshot, home_pitcher_id_live, side="home")

    away_used_pitchers = get_used_pitchers_from_snapshot(snapshot, side="away")
    home_used_pitchers = get_used_pitchers_from_snapshot(snapshot, side="home")

    results = []
    for _ in range(n_sims):
        g = simulate_remaining_game_from_state(
            away_lineup_ids=away_lineup_ids,
            home_lineup_ids=home_lineup_ids,
            away_team_code=snapshot["away_abbr"],
            home_team_code=snapshot["home_abbr"],
            away_pitcher_state=away_pitcher_state_live.copy(),
            home_pitcher_state=home_pitcher_state_live.copy(),
            away_used_pitchers=set(away_used_pitchers),
            home_used_pitchers=set(home_used_pitchers),
            away_score_start=norm["away_score"],
            home_score_start=norm["home_score"],
            start_inning=norm["start_inning"],
            start_half=norm["start_half"],
            away_lineup_pos=away_lineup_pos,
            home_lineup_pos=home_lineup_pos
        )
        results.append(g)

    results_df = pd.DataFrame(results)

    return {
        "results_df": results_df,
        "away_avg_runs": results_df["away_runs"].mean(),
        "home_avg_runs": results_df["home_runs"].mean(),
        "away_win_pct": (results_df["away_runs"] > results_df["home_runs"]).mean(),
        "home_win_pct": (results_df["home_runs"] > results_df["away_runs"]).mean(),
        "unresolved_tie_pct": (results_df["away_runs"] == results_df["home_runs"]).mean(),
        "normalized_start_state": norm
    }

In [132]:
# ============================================================
# CELL O: display pregame + live forecast cleanly
# ============================================================

def projected_score_from_result(sim_result):
    away_proj = int(round(sim_result["away_avg_runs"]))
    home_proj = int(round(sim_result["home_avg_runs"]))
    return away_proj, home_proj

def show_pregame_and_live_forecasts(snapshot, lineup_info, pregame_result, live_result):
    away_abbr = snapshot["away_abbr"]
    home_abbr = snapshot["home_abbr"]

    pre_away, pre_home = projected_score_from_result(pregame_result)
    live_away, live_home = projected_score_from_result(live_result)

    print("=== PREGAME FORECAST (before first pitch) ===")
    print(f"Projected score: {away_abbr} {pre_away} - {home_abbr} {pre_home}")
    print(f"Average runs: {away_abbr} {pregame_result['away_avg_runs']:.2f} | {home_abbr} {pregame_result['home_avg_runs']:.2f}")
    print(f"Win %: {away_abbr} {pregame_result['away_win_pct']*100:.1f}% | {home_abbr} {pregame_result['home_win_pct']*100:.1f}%")
    print()

    print("=== LIVE UPDATED FINAL FORECAST ===")
    print("Starting from:", live_result["normalized_start_state"])
    print(f"Current score: {away_abbr} {snapshot.get('away_score')} - {home_abbr} {snapshot.get('home_score')}")
    print(f"Projected final: {away_abbr} {live_away} - {home_abbr} {live_home}")
    print(f"Average final runs: {away_abbr} {live_result['away_avg_runs']:.2f} | {home_abbr} {live_result['home_avg_runs']:.2f}")
    print(f"Win % from here: {away_abbr} {live_result['away_win_pct']*100:.1f}% | {home_abbr} {live_result['home_win_pct']*100:.1f}%")

In [134]:
# ============================================================
# BLOCK 1: PREGAME PROJECTED SCORE
# ============================================================

def run_pregame_projection(game_pk, n_sims=150):
    snapshot = get_game_snapshot(game_pk)
    lineup_info = resolve_game_lineups(snapshot)

    away_lineup_ids = lineup_df_to_ids(lineup_info["away_lineup"])
    home_lineup_ids = lineup_df_to_ids(lineup_info["home_lineup"])

    validate_lineup_ids(away_lineup_ids, "Away")
    validate_lineup_ids(home_lineup_ids, "Home")

    # try probable starters first
    away_pitcher_id = probable_pitcher_id_from_name(snapshot.get("away_probable_pitcher"))
    home_pitcher_id = probable_pitcher_id_from_name(snapshot.get("home_probable_pitcher"))

    # if probable starters missing, try current/fallback pitcher
    if away_pitcher_id is None:
        away_pitcher_id = get_current_or_fallback_pitcher_id(snapshot, side="away")
    if home_pitcher_id is None:
        home_pitcher_id = get_current_or_fallback_pitcher_id(snapshot, side="home")

    if away_pitcher_id is None or home_pitcher_id is None:
        raise ValueError("Could not resolve one or both pitchers for pregame projection.")

    pregame_result = simulate_matchup_with_pitching(
        away_lineup_ids=away_lineup_ids,
        home_lineup_ids=home_lineup_ids,
        away_pitcher_id=away_pitcher_id,
        home_pitcher_id=home_pitcher_id,
        away_team_code=snapshot["away_abbr"],
        home_team_code=snapshot["home_abbr"],
        n_sims=n_sims
    )

    away_proj = int(round(pregame_result["away_avg_runs"]))
    home_proj = int(round(pregame_result["home_avg_runs"]))

    print("=== PREGAME PROJECTED SCORE ===")
    print(f"Matchup: {snapshot['away_abbr']} @ {snapshot['home_abbr']}")
    print(f"Away pitcher: {snapshot.get('away_probable_pitcher')}")
    print(f"Home pitcher: {snapshot.get('home_probable_pitcher')}")
    print(f"Away lineup source: {lineup_info['away_source']}")
    print(f"Home lineup source: {lineup_info['home_source']}")
    print()
    print(f"Projected score: {snapshot['away_abbr']} {away_proj} - {snapshot['home_abbr']} {home_proj}")
    print(f"Average runs: {snapshot['away_abbr']} {pregame_result['away_avg_runs']:.2f} | {snapshot['home_abbr']} {pregame_result['home_avg_runs']:.2f}")
    print(f"Win %: {snapshot['away_abbr']} {pregame_result['away_win_pct']*100:.1f}% | {snapshot['home_abbr']} {pregame_result['home_win_pct']*100:.1f}%")

    return {
        "snapshot": snapshot,
        "lineup_info": lineup_info,
        "result": pregame_result
    }

In [135]:
# ============================================================
# BLOCK 2: LIVE UPDATED FINAL SCORE
# ============================================================

def run_live_projection(game_pk, n_sims=150):
    snapshot = get_game_snapshot(game_pk)
    lineup_info = resolve_game_lineups(snapshot)

    away_lineup_ids = lineup_df_to_ids(lineup_info["away_lineup"])
    home_lineup_ids = lineup_df_to_ids(lineup_info["home_lineup"])

    validate_lineup_ids(away_lineup_ids, "Away")
    validate_lineup_ids(home_lineup_ids, "Home")

    norm = normalize_next_half_inning_state(snapshot)

    away_lineup_pos, home_lineup_pos = next_lineup_positions_from_feed(
        snapshot,
        away_lineup_ids,
        home_lineup_ids
    )

    away_pitcher_id_live = get_current_or_fallback_pitcher_id(snapshot, side="away")
    home_pitcher_id_live = get_current_or_fallback_pitcher_id(snapshot, side="home")

    if away_pitcher_id_live is None or home_pitcher_id_live is None:
        raise ValueError("Could not resolve one or both current pitchers for live projection.")

    away_pitcher_state_live = init_live_pitcher_state(snapshot, away_pitcher_id_live, side="away")
    home_pitcher_state_live = init_live_pitcher_state(snapshot, home_pitcher_id_live, side="home")

    away_used_pitchers = get_used_pitchers_from_snapshot(snapshot, side="away")
    home_used_pitchers = get_used_pitchers_from_snapshot(snapshot, side="home")

    results = []
    for _ in range(n_sims):
        g = simulate_remaining_game_from_state(
            away_lineup_ids=away_lineup_ids,
            home_lineup_ids=home_lineup_ids,
            away_team_code=snapshot["away_abbr"],
            home_team_code=snapshot["home_abbr"],
            away_pitcher_state=away_pitcher_state_live.copy(),
            home_pitcher_state=home_pitcher_state_live.copy(),
            away_used_pitchers=set(away_used_pitchers),
            home_used_pitchers=set(home_used_pitchers),
            away_score_start=norm["away_score"],
            home_score_start=norm["home_score"],
            start_inning=norm["start_inning"],
            start_half=norm["start_half"],
            away_lineup_pos=away_lineup_pos,
            home_lineup_pos=home_lineup_pos
        )
        results.append(g)

    live_df = pd.DataFrame(results)

    live_result = {
        "results_df": live_df,
        "away_avg_runs": live_df["away_runs"].mean(),
        "home_avg_runs": live_df["home_runs"].mean(),
        "away_win_pct": (live_df["away_runs"] > live_df["home_runs"]).mean(),
        "home_win_pct": (live_df["home_runs"] > live_df["away_runs"]).mean(),
        "unresolved_tie_pct": (live_df["away_runs"] == live_df["home_runs"]).mean(),
        "normalized_start_state": norm
    }

    away_proj = int(round(live_result["away_avg_runs"]))
    home_proj = int(round(live_result["home_avg_runs"]))

    print("=== LIVE UPDATED FINAL SCORE ===")
    print(f"Matchup: {snapshot['away_abbr']} @ {snapshot['home_abbr']}")
    print(f"Current score: {snapshot['away_abbr']} {snapshot.get('away_score')} - {snapshot['home_abbr']} {snapshot.get('home_score')}")
    print(f"Update point: inning {norm['start_inning']} {norm['start_half']}")
    print(f"Current away pitcher ID: {away_pitcher_id_live}")
    print(f"Current home pitcher ID: {home_pitcher_id_live}")
    print()
    print(f"Projected final: {snapshot['away_abbr']} {away_proj} - {snapshot['home_abbr']} {home_proj}")
    print(f"Average final runs: {snapshot['away_abbr']} {live_result['away_avg_runs']:.2f} | {snapshot['home_abbr']} {live_result['home_avg_runs']:.2f}")
    print(f"Win % from here: {snapshot['away_abbr']} {live_result['away_win_pct']*100:.1f}% | {snapshot['home_abbr']} {live_result['home_win_pct']*100:.1f}%")

    return {
        "snapshot": snapshot,
        "lineup_info": lineup_info,
        "result": live_result
    }

In [136]:
# ============================================================
# SELECTOR: PREGAME PROJECTION
# ============================================================

def launch_pregame_projection_selector(game_date=None, default_sims=150):
    games_df = get_daily_games(game_date=game_date)

    if games_df.empty:
        print("No MLB games found for that date.")
        return None

    dropdown = widgets.Dropdown(
        options=[(str(row["label"]), int(row["gamePk"])) for _, row in games_df.iterrows()],
        description="Pregame:",
        layout=widgets.Layout(width="900px")
    )

    sims_box = widgets.IntText(
        value=default_sims,
        description="Sims:",
        layout=widgets.Layout(width="180px")
    )

    run_btn = widgets.Button(description="Run Pregame Projection", button_style="primary")
    output = widgets.Output()

    def on_click(_):
        with output:
            clear_output()

            game_pk = dropdown.value
            n_sims = int(sims_box.value)

            print(f"Running pregame projection for gamePk={game_pk} with {n_sims} sims...\n")

            try:
                pregame_out = run_pregame_projection(game_pk, n_sims=n_sims)
                return pregame_out
            except Exception as e:
                print("Pregame projection failed:")
                print(e)

    run_btn.on_click(on_click)

    display(widgets.VBox([
        widgets.HBox([dropdown, sims_box]),
        run_btn,
        output
    ]))

    return games_df

In [137]:
# ============================================================
# SELECTOR: LIVE PROJECTION
# ============================================================

def launch_live_projection_selector(game_date=None, default_sims=150):
    games_df = get_daily_games(game_date=game_date)

    if games_df.empty:
        print("No MLB games found for that date.")
        return None

    dropdown = widgets.Dropdown(
        options=[(str(row["label"]), int(row["gamePk"])) for _, row in games_df.iterrows()],
        description="Live:",
        layout=widgets.Layout(width="900px")
    )

    sims_box = widgets.IntText(
        value=default_sims,
        description="Sims:",
        layout=widgets.Layout(width="180px")
    )

    run_btn = widgets.Button(description="Run Live Projection", button_style="success")
    output = widgets.Output()

    def on_click(_):
        with output:
            clear_output()

            game_pk = dropdown.value
            n_sims = int(sims_box.value)

            print(f"Running live projection for gamePk={game_pk} with {n_sims} sims...\n")

            try:
                live_out = run_live_projection(game_pk, n_sims=n_sims)
                return live_out
            except Exception as e:
                print("Live projection failed:")
                print(e)

    run_btn.on_click(on_click)

    display(widgets.VBox([
        widgets.HBox([dropdown, sims_box]),
        run_btn,
        output
    ]))

    return games_df

In [ ]:
# ============================================================
# BOTH SELECTORS
# ============================================================

pregame_games_today = launch_pregame_projection_selector()
live_games_today = launch_live_projection_selector()